In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from src.utils import *
import src.prompt as prompt
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [ ]:
data_name_list = ["BZ5", "BZ9", "BZ14"]
results_dict = {}

In [ ]:
for data_name in data_name_list:
    config = load_config("configs/config_cluster_singlecell.yaml")
    config.data_name = data_name
    config.refresh_paths()

    # --- Load data ---
    data_path = str(dataset_dir("starmap", config.data_name))
    x_data_name = "data.csv"  
    index_col = 0
    adata = sc.read_csv(f"{data_path}/{x_data_name}", first_column_names=True)
    celltype_data = pd.read_csv(f"{data_path}/celltype.csv", index_col=index_col)  # Assuming first column is index
    celltype_data.columns = ["cell_type"] 
    pos_data = pd.read_csv(f"{data_path}/pos.csv", index_col=index_col)
    pos_data.columns = ['x', 'y']
    domain_data = pd.read_csv(f"{data_path}/domain.csv", index_col=index_col)
    domain_data.columns = [config.name_truth]
    adata.obs = adata.obs.join([celltype_data, pos_data, domain_data])

    # clean the cell ID to save token
    adata.obs_names = list(range(len(adata)))
    adata.obs_names = adata.obs_names.astype(str)

    pos_data = adata.obs[['x', 'y']]
    celltype_data = adata.obs[['cell_type']]
    domain_data = adata.obs[[config.name_truth]]

    sc.pp.filter_genes(adata, min_cells=5)
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    sc.pp.scale(adata)

    # Initialize a dictionary to store the top genes per cell type
    top_genes_per_cell_type = {}

    for cell_type in adata.obs['cell_type'].unique():
        # Subset the data for the current cell type
        adata_subset = adata[adata.obs['cell_type'] == cell_type].copy()
        if len(adata_subset) < 30:
            continue

        
        # Compute highly variable genes within the subset
        sc.pp.highly_variable_genes(
            adata_subset,
            n_top_genes=5,
            flavor='seurat',
            subset=False,
            layer=None,
            inplace=True
        )
        
        # Retrieve the top 5 highly variable genes
        top_genes = adata_subset.var.loc[adata_subset.var['highly_variable'], :].index.tolist()
        
        # Store the results in the dictionary
        top_genes_per_cell_type[cell_type] = top_genes

    # Convert the dictionary to a DataFrame for better visualization
    top_genes_df = pd.DataFrame.from_dict(top_genes_per_cell_type, orient='index').transpose()

    # get all the top genes
    top_genes = list(set(top_genes_df.values.flatten()))

    kmeansBoth_ari = []
    kmeansBoth_nmi = []
    kmeans_ari = [] 
    kmeans_nmi = []
    kmeans_genes_ari = []
    kmeans_genes_nmi = []

    for r in range(10, 2000, 10):
        adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
        # add diagonal to the adj_matrix
        adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

        # --- Generate one-hot encoded matrix ---
        one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
        one_hot_matrix = one_hot_df.values
        one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

        # --- Calculate neighbor counts ---
        neighbor_count = adj_matrix.dot(one_hot_matrix)
        n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
        # Convert n_neighbors to a column vector for element-wise division
        n_neighbors_col = n_neighbors.reshape(-1, 1)
        # Perform element-wise division between neighbor_count and n_neighbors_col
        neighbor_matrix_normalized = neighbor_count / n_neighbors_col

        neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                                    index=celltype_data.index, 
                                    columns=one_hot_df.columns)
        # --- Calculate neighbor genes ---
        neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

        # Perform element-wise division between neighbor_count and n_neighbors_col
        neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

        neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                                    index=adata.obs_names, 
                                    columns=top_genes)
        
        neighbor_scaled_df = neighbor_normalized_df.join(neighbor_normalized_df_genes).copy()

        # kmeans with both neighbor count and neighbor genes
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_scaled_df)
        kmeansBoth_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kmeansBoth_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
        # kmeans with neighbor count
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_normalized_df)
        kmeans_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kmeans_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
        # kmeans with neighbor genes
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_normalized_df_genes)
        kmeans_genes_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kmeans_genes_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
    results_dict[data_name] = {"kmeans_ari": kmeans_ari, "kmeans_genes_ari": kmeans_genes_ari, "kmeansBoth_ari": kmeansBoth_ari, 
                               "kmeans_nmi": kmeans_nmi, "kmeans_genes_nmi": kmeans_genes_nmi, "kmeansBoth_nmi": kmeansBoth_nmi}



In [ ]:
# plot results_dict in three subplots
fig, axs = plt.subplots(3, 1, figsize=(10, 15))
for i, data_name in enumerate(data_name_list):
    axs[i].plot(range(10, 2000, 10), results_dict[data_name]["kmeansBoth_ari"], label="kmeansBoth")
    axs[i].plot(range(10, 2000, 10), results_dict[data_name]["kmeans_ari"], label="kmeans")
    axs[i].plot(range(10, 2000, 10), results_dict[data_name]["kmeans_genes_ari"], label="kmeans_genes")
    axs[i].legend()
    axs[i].set_title(data_name)
plt.show()

In [ ]:
# save the results_dict
import pickle
with open('examples/results/kmeans_results/kmeans_r_starmap_results_dict.pkl', 'wb') as f:
    pickle.dump(results_dict, f)




In [ ]:
# load the result_dict
import pickle

with open('examples/results/kmeans_results/kmeans_r_starmap_results_dict.pkl', 'rb') as f:
    results_dict = pickle.load(f)



In [ ]:
results_dict[data_name].keys()

In [ ]:
# Collect NMI scores at r=100
r700_index = range(10, 1000, 10).index(700)  # Find index corresponding to r=100
kmeansBoth_r700 = []
kmeans_r700 = []
kmeans_genes_r700 = []

for data_name in data_name_list:
	kmeansBoth_r700.append(results_dict[data_name]["kmeansBoth_nmi"][r700_index])
	kmeans_r700.append(results_dict[data_name]["kmeans_nmi"][r700_index])
	kmeans_genes_r700.append(results_dict[data_name]["kmeans_genes_nmi"][r700_index])


# Create boxplot
plt.figure(figsize=(8, 6))
box_data = [kmeansBoth_r700, kmeans_r700, kmeans_genes_r700]
plt.boxplot(box_data, labels=['KMeans Both', 'KMeans', 'KMeans Genes'])
plt.title('NMI Scores at r=700 Across Datasets')
plt.ylabel('NMI Score')
plt.grid(True, alpha=0.3)

# Add individual points for each dataset
for i, data in enumerate(box_data, 1):
    plt.scatter([i] * len(data), data, alpha=0.6, color='red', 
                marker='o', s=50, zorder=3)

plt.show()